# **MODEL EXPERIMENTATION**
with GCP Integration


by Jack Phelan

In [30]:
#imports 
import sys

sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scripts.plot_utils import (
    set_theme,
    plot_grid,
    plot_barplot,
    plot_barplot_grid,
    plot_histogram,
    plot_numeric_x_numeric_grid,
    plot_numeric_x_across_categories_grid,
    plot_all_numeric_by_base_category_grid,
    plot_categorical_x_categorical_grid,
)
from scripts.gcs_utils import (
    log_dataset_to_gcs,
    log_pipeline_run,    
)
from kfp.v2 import dsl
from kfp.v2.dsl import component, Output, Dataset, Input, Model, Metrics, Artifact
from google.cloud import aiplatform

In [31]:
# variable declarations
TARGET_COL = "readmission_within_30_days"
ID_COL = "patient_id"
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmission-bucketv2"


---
## KFP Preprocessing Pipeline

Self-contained KFP v2 components for the preprocessing pipeline.  
Order: `load_validate_data` → `split_data` → `oversample_training` → `fit_apply_preprocessing` → `apply_preprocessing`

In [32]:
@component(
    packages_to_install=["pandas", "numpy", "fsspec", "gcsfs"],
    base_image="python:3.10-slim",
)
def load_validate_data(
    input_dataset_path: str,
    output_dataset: Output[Dataset],
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    import pandas as pd
    import numpy as np

    def format_columns(df: pd.DataFrame) -> pd.DataFrame:
        df_transformed = df.copy()

        if "PatientID" in df_transformed.columns:
            df_transformed = df_transformed.rename(
                columns={"PatientID": "patient_id"}
            )

        df_transformed.columns = (
            df_transformed.columns.str.strip().str.lower().str.replace(" ", "_")
        )
        df_transformed.columns = df_transformed.columns.str.replace("(", "").str.replace(
            ")", ""
        )
        return df_transformed  

    df = pd.read_csv(input_dataset_path)

    if df.empty:
        raise ValueError("Input dataset is empty")

    print(f"Dataset shape: {df.shape}")

    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
    df = format_columns(df)

    missing = [c for c in [target_col, id_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    print("Target distribution:")
    print(df[target_col].value_counts(dropna=False))

    if df[target_col].dropna().nunique() < 2:
        raise ValueError(f"Target column '{target_col}' must contain at least two classes")

    df.to_csv(output_dataset.path, index=False)


In [33]:
@component(
    packages_to_install=["pandas", "scikit-learn"],
    base_image="python:3.10-slim",
)
def split_data(
    input_dataset: Input[Dataset],
    train_dataset: Output[Dataset],
    validation_dataset: Output[Dataset],
    split_metrics: Output[Metrics],
    test_size: float = 0.2,
    random_state: int = 42,
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    from sklearn.model_selection import train_test_split
    import pandas as pd

    df = pd.read_csv(input_dataset.path)

    X = df.drop(columns=[target_col, id_col])
    y = df[target_col]
    ids = df[id_col]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    train_df = pd.concat([X_train, y_train, ids.loc[X_train.index]], axis=1)
    val_df = pd.concat([X_val, y_val, ids.loc[X_val.index]], axis=1)

    train_df.to_csv(train_dataset.path, index=False)
    val_df.to_csv(validation_dataset.path, index=False)

    split_metrics.log_metric("train_size", len(train_df))
    split_metrics.log_metric("val_size", len(val_df))
    split_metrics.log_metric("train_positive_rate", float(train_df[target_col].mean()))
    split_metrics.log_metric("val_positive_rate", float(val_df[target_col].mean()))

    print(f"Train shape: {train_df.shape}, Val shape: {val_df.shape}")
    print("Train target distribution:")
    print(train_df[target_col].value_counts(normalize=True))
    print("Val target distribution:")
    print(val_df[target_col].value_counts(normalize=True))


In [34]:
@component(
    packages_to_install=["pandas", "numpy", "scikit-learn", "imbalanced-learn"],
    base_image="python:3.10-slim",
)
def oversample_training(
    input_dataset: Input[Dataset],
    output_dataset: Output[Dataset],
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
    random_state: int = 42,
):
    import pandas as pd
    from imblearn.over_sampling import RandomOverSampler

    df = pd.read_csv(input_dataset.path)

    print(f"Input shape: {df.shape}")
    print("Class distribution BEFORE oversampling:")
    print(df[target_col].value_counts(dropna=False))

    missing = [c for c in [target_col, id_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    X = df.drop(columns=[target_col])
    y = df[target_col]

    ros = RandomOverSampler(random_state=random_state)
    X_resampled, y_resampled = ros.fit_resample(X, y)

    resampled_df = X_resampled.copy()
    resampled_df[target_col] = y_resampled

    print("Class distribution AFTER oversampling:")
    print(resampled_df[target_col].value_counts(dropna=False))
    print(f"Output shape: {resampled_df.shape}")

    resampled_df.to_csv(output_dataset.path, index=False)


In [35]:
@component(
    packages_to_install=["pandas", "numpy", "scikit-learn", "joblib"],
    base_image="python:3.10-slim",
)
def fit_apply_preprocessing_v1(
    input_dataset: Input[Dataset],
    output_dataset: Output[Dataset],
    preprocessing_artifacts: Output[Artifact],
    preprocessing_metrics: Output[Metrics],
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    import os
    import joblib
    import numpy as np
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X["age_group"] = pd.cut(
            X["age"],
            bins=[0, 18, 25, 40, 65, 80, np.inf],
            labels=["0-18", "19-25", "26-40", "41-65", "66-80", "81+"],
            right=False,
        ).astype(str)
        X = X.drop(columns=["age"])
        X["medications_prescribed"] = (
            X["medications_prescribed"].replace("", pd.NA).astype(float).apply(lambda x: 1 if x > 0 else 0)
        )
        X["number_of_prior_visits"] = X["number_of_prior_visits"].replace("", pd.NA).astype(float)
        X["length_of_stay_score"] = X["length_of_stay"].apply(
            lambda x: 1 if x <= 1 else (2 if x <= 2 else (3 if x <= 3 else (4 if x <= 6 else (5 if x <= 14 else 7))))
        )
        X = X.drop(columns=["length_of_stay"])
        return X

    df = pd.read_csv(input_dataset.path)
    print(f"Input training shape: {df.shape}")

    ids = df[[id_col]].copy()
    y = df[[target_col]].copy()
    X = df.drop(columns=[id_col, target_col]).copy()

    X = engineer_features(X)

    num_cols = ["height_m", "bmi", "adjusted_weight_kg", "number_of_prior_visits", "length_of_stay_score"]
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]),
                num_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("encode", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
                ]),
                cat_cols,
            ),
        ],
        remainder="passthrough",
    )

    X_transformed = preprocessor.fit_transform(X)
    feature_names = preprocessor.get_feature_names_out()
    X_df = pd.DataFrame(X_transformed, columns=feature_names)

    prepared_df = pd.concat(
        [ids.reset_index(drop=True), X_df, y.reset_index(drop=True)], axis=1
    )
    print(f"Output training shape: {prepared_df.shape}")
    prepared_df.to_csv(output_dataset.path, index=False)

    os.makedirs(preprocessing_artifacts.path, exist_ok=True)
    joblib.dump(preprocessor, os.path.join(preprocessing_artifacts.path, "preprocessor.joblib"))
    print("Preprocessing artifact saved:", os.listdir(preprocessing_artifacts.path))

    preprocessing_metrics.log_metric("input_features", int(X.shape[1]))
    preprocessing_metrics.log_metric("output_features", int(X_df.shape[1]))
    preprocessing_metrics.log_metric("train_samples", int(len(prepared_df)))


In [36]:
@component(
    packages_to_install=["pandas", "numpy", "scikit-learn", "joblib"],
    base_image="python:3.10-slim",
)
def apply_preprocessing_v1(
    input_dataset: Input[Dataset],
    preprocessing_artifacts: Input[Artifact],
    output_dataset: Output[Dataset],
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    import os
    import joblib
    import numpy as np
    import pandas as pd

    def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X["age_group"] = pd.cut(
            X["age"],
            bins=[0, 18, 25, 40, 65, 80, np.inf],
            labels=["0-18", "19-25", "26-40", "41-65", "66-80", "81+"],
            right=False,
        ).astype(str)
        X = X.drop(columns=["age"])
        X["medications_prescribed"] = (
            X["medications_prescribed"].replace("", pd.NA).astype(float).apply(lambda x: 1 if x > 0 else 0)
        )
        X["number_of_prior_visits"] = X["number_of_prior_visits"].replace("", pd.NA).astype(float)
        X["length_of_stay_score"] = X["length_of_stay"].apply(
            lambda x: 1 if x <= 1 else (2 if x <= 2 else (3 if x <= 3 else (4 if x <= 6 else (5 if x <= 14 else 7))))
        )
        X = X.drop(columns=["length_of_stay"])
        return X

    df = pd.read_csv(input_dataset.path)
    print(f"Input validation shape: {df.shape}")

    preprocessor_path = os.path.join(preprocessing_artifacts.path, "preprocessor.joblib")
    if not os.path.exists(preprocessor_path):
        raise ValueError(f"Missing preprocessor artifact: {preprocessor_path}")

    preprocessor = joblib.load(preprocessor_path)

    ids = df[[id_col]].copy()
    y = df[[target_col]].copy()
    X = df.drop(columns=[id_col, target_col]).copy()

    X = engineer_features(X)

    X_transformed = preprocessor.transform(X)
    feature_names = preprocessor.get_feature_names_out()
    X_df = pd.DataFrame(X_transformed, columns=feature_names)

    prepared_df = pd.concat(
        [ids.reset_index(drop=True), X_df, y.reset_index(drop=True)], axis=1
    )
    print(f"Output validation shape: {prepared_df.shape}")
    prepared_df.to_csv(output_dataset.path, index=False)


In [37]:
from kfp.dsl import pipeline
from kfp import compiler


@pipeline(name="readmissions-preprocessing-pipeline")
def preprocessing_pipeline(
    training_dataset_path: str,
    test_size: float = 0.2,
    random_state: int = 42,
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    # 1. Load & validate
    validated = (
        load_validate_data(
            input_dataset_path=training_dataset_path,
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )

    # 2. Train / val split
    split = (
        split_data(
            input_dataset=validated.outputs["output_dataset"],
            test_size=test_size,
            random_state=random_state,
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )

    # 3. Oversample training split only
    oversampled = (
        oversample_training(
            input_dataset=split.outputs["train_dataset"],
            random_state=random_state,
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )

    # 4. Fit preprocessing on oversampled training data & transform
    preprocessed_train = (
        fit_apply_preprocessing_v1(
            input_dataset=oversampled.outputs["output_dataset"],
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("2")
        .set_memory_limit("8G")
    )

    # 5. Apply saved preprocessing artifacts to validation split
    preprocessed_val = (
        apply_preprocessing_v1(
            input_dataset=split.outputs["validation_dataset"],
            preprocessing_artifacts=preprocessed_train.outputs["preprocessing_artifacts"],
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )


PREPROCESSING_PIPELINE_JSON = "readmissions_preprocessing_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=preprocessing_pipeline,
    package_path=PREPROCESSING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {PREPROCESSING_PIPELINE_JSON}")


Pipeline compiled to readmissions_preprocessing_pipeline.json


---
## KFP Training Pipeline

`train_model` and `evaluate_model` components extending the preprocessing pipeline.  
Order: `...preprocessing...` → `train_model` → `evaluate_model`

- **`model_type`**: `"logistic"` | `"random_forest"` | `"xgboost"`  
- **`hyperparams_json`**: JSON string of kwargs passed to the chosen estimator (e.g. `'{"n_estimators": 200, "max_depth": 5}'`)

In [38]:
@component(
    packages_to_install=["pandas", "scikit-learn", "joblib", "xgboost"],
    base_image="python:3.10-slim",
)
def train_model(
    train_dataset: Input[Dataset],
    model_artifact: Output[Model],
    train_metrics: Output[Metrics],
    model_type: str = "xgboost",
    hyperparams_json: str = "{}",
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    import json
    import joblib
    import pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, f1_score
    from xgboost import XGBClassifier

    df = pd.read_csv(train_dataset.path)
    X = df.drop(columns=[id_col, target_col])
    y = df[target_col]

    hyperparams = json.loads(hyperparams_json)

    if model_type == "logistic":
        model = LogisticRegression(max_iter=1000, **hyperparams)
    elif model_type == "random_forest":
        model = RandomForestClassifier(**hyperparams)
    elif model_type == "xgboost":
        model = XGBClassifier(eval_metric="logloss", **hyperparams)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.fit(X, y)

    train_preds = model.predict(X)
    train_proba = model.predict_proba(X)[:, 1]

    train_metrics.log_metric("train_roc_auc", float(roc_auc_score(y, train_proba)))
    train_metrics.log_metric("train_f1", float(f1_score(y, train_preds)))
    train_metrics.log_metric("model_type", model_type)

    joblib.dump(model, model_artifact.path)
    print(f"Model trained and saved: {model_type}")


In [39]:
@component(
    packages_to_install=["pandas", "scikit-learn", "joblib", "xgboost"],
    base_image="python:3.10-slim",
)
def evaluate_model(
    val_dataset: Input[Dataset],
    model_artifact: Input[Model],
    eval_metrics: Output[Metrics],
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    import joblib
    import pandas as pd
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        f1_score,
        precision_score,
        recall_score,
    )

    df = pd.read_csv(val_dataset.path)
    X = df.drop(columns=[id_col, target_col])
    y = df[target_col]

    model = joblib.load(model_artifact.path)
    preds = model.predict(X)
    proba = model.predict_proba(X)[:, 1]

    roc_auc = roc_auc_score(y, proba)
    pr_auc = average_precision_score(y, proba)
    f1 = f1_score(y, preds)
    precision = precision_score(y, preds)
    recall = recall_score(y, preds)

    eval_metrics.log_metric("val_roc_auc", float(roc_auc))
    eval_metrics.log_metric("val_pr_auc", float(pr_auc))
    eval_metrics.log_metric("val_f1", float(f1))
    eval_metrics.log_metric("val_precision", float(precision))
    eval_metrics.log_metric("val_recall", float(recall))

    print(
        f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | "
        f"F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}"
    )


In [40]:
@pipeline(name="readmissions-training-pipeline")
def training_pipeline(
    training_dataset_path: str,
    model_type: str = "xgboost",
    hyperparams_json: str = "{}",
    test_size: float = 0.2,
    random_state: int = 42,
    target_col: str = "readmission_within_30_days",
    id_col: str = "patient_id",
):
    # 1. Load & validate
    validated = (
        load_validate_data(
            input_dataset_path=training_dataset_path,
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )

    # 2. Train / val split
    split = (
        split_data(
            input_dataset=validated.outputs["output_dataset"],
            test_size=test_size,
            random_state=random_state,
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )

    # 3. Oversample training split only
    oversampled = (
        oversample_training(
            input_dataset=split.outputs["train_dataset"],
            random_state=random_state,
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )

    # 4. Fit preprocessing on oversampled training data & transform
    preprocessed_train = (
        fit_apply_preprocessing_v1(
            input_dataset=oversampled.outputs["output_dataset"],
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("2")
        .set_memory_limit("8G")
    )

    # 5. Apply saved preprocessing artifacts to validation split
    preprocessed_val = (
        apply_preprocessing_v1(
            input_dataset=split.outputs["validation_dataset"],
            preprocessing_artifacts=preprocessed_train.outputs["preprocessing_artifacts"],
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )

    # 6. Train model
    trained = (
        train_model(
            train_dataset=preprocessed_train.outputs["output_dataset"],
            model_type=model_type,
            hyperparams_json=hyperparams_json,
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("2")
        .set_memory_limit("8G")
    )

    # 7. Evaluate on validation set
    (
        evaluate_model(
            val_dataset=preprocessed_val.outputs["output_dataset"],
            model_artifact=trained.outputs["model_artifact"],
            target_col=target_col,
            id_col=id_col,
        )
        .set_cpu_limit("1")
        .set_memory_limit("4G")
    )


TRAINING_PIPELINE_JSON = "readmissions_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path=TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {TRAINING_PIPELINE_JSON}")


Pipeline compiled to readmissions_training_pipeline.json
